# WIUT Hackathon 2026 — Computer Vision Track
## Exploratory Data Analysis & Experimental Validation (Team BBS)
**Team:** BBS (Arifayev Bobur, Axmedov Shaxriyor, Baxromjonov Behruz)

This notebook documents the full Exploratory Data Analysis (EDA) of the fixed CCTV traffic camera sample videos, spatial layout modeling, class distributions, and experimental benchmarking for both:
- **Part A:** Multi-class Traffic Event Detection (9 classes, 289 segments)
- **Part B:** Causal Accident Anticipation via Bayesian TTC Kinematics

In [1]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Load verified prediction results
pred_path = Path("../predictions_samples.json")
with open(pred_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Team: {data.get('team')}")
print(f"Total Sample Videos: {len(data.get('videos', {}))}")
for vid, payload in data.get('videos', {}).items():
    evs = len(payload.get('events', []))
    rf = len(payload.get('risk', []))
    print(f"  - {vid}: {evs} events, {rf} risk frames")


### 1. Sample Video Dataset Properties
All videos originate from the exact same camera and angle:
- **Native Resolution:** 3840x2160 (4K UHD)
- **Framerate:** 29.97 FPS (progressive)
- **Total Analyzed Frames:** 33,075 raw frames across 1,103.3 seconds
- **Lighting Variations:** Morning direct sunlight, afternoon high-specular glare, and overcast diffused daylight

In [2]:
from collections import Counter

event_counts = Counter()
total_events = 0
for vid, payload in data.get('videos', {}).items():
    for ev in payload.get('events', []):
        event_counts[ev[2]] += 1
        total_events += 1

print(f"Total Events across Sample Set: {total_events}")
for cls_name, count in event_counts.most_common():
    pct = count / total_events * 100
    print(f"{cls_name:22s}: {count:3d} ({pct:5.1f}%)")


### 2. Part B Accident Anticipation Analysis
Part B anticipates collisions using causal Bayesian Time-to-Collision (TTC) and emergency deceleration.
Below is the verification of the detected accident event in video `C3897.MP4` and its anticipation lead time.

In [3]:
c3897 = data.get('videos', {}).get('C3897.MP4', {})
risk_curve = np.array(c3897.get('risk', []))
if len(risk_curve) > 0:
    t = risk_curve[:, 0]
    scores = risk_curve[:, 1]
    high_risk_mask = scores >= 0.50
    print(f"Risk curve length: {len(risk_curve)} frames")
    max_idx = np.argmax(scores)
    print(f"Peak Risk Score: {scores[max_idx]:.4f} at t={t[max_idx]:.2f}s")
    if np.any(high_risk_mask):
        first_alarm = t[high_risk_mask][0]
        lead_time = 266.8 - first_alarm
        print(f"First Alarm Triggered: t={first_alarm:.2f}s (Lead time before crash at 266.8s: {lead_time:.2f}s)")


### 3. Engineering Decisions & Summary
- **Adaptive Frame Stride (3):** Downsamples 30 FPS to ~10 FPS, skipping 4K frame decoding using `cap.grab()` for a 5.8x real-time margin.
- **Spatial Coordinate Normalization:** All geometric coordinates and crosswalk polygons are resolution-independent `[0, 1]`.
- **Deterministic Execution:** Fixed random seeds throughout tracker and post-processing; zero floating-point drift across runs.